In [13]:
import os
import sys
sys.path.append(os.path.abspath('..'))

In [23]:
from src.utils import load_and_vectorize_data
import torch.nn as nn
import torch.optim as optim
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

In [15]:
encoder_input, decoder_input, decoder_target, src_vectorizer, trg_vectorizer, max_src_len, max_trg_len = load_and_vectorize_data()

src_vectorizer.vocab_size = len(src_vectorizer.get_vocabulary())
trg_vectorizer.vocab_size = len(trg_vectorizer.get_vocabulary())
trg_vectorizer.word_to_index = {word: i for i, word in enumerate(trg_vectorizer.get_vocabulary())}

Vectorization Ready. English Vocab: 3355, French Vocab: 7801
Max Source Length: 5, Max Target Length: 12


In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, src_vectorizer, trg_vectorizer):
        super().__init__()

        self.state_size = 128
        self.embedding_dim = 64
        
        # Save this to class so we can access word_to_index in forward()
        self.trg_vectorizer = trg_vectorizer 

        self.encoder_lstm = nn.LSTM(input_size=self.embedding_dim, hidden_size=self.state_size, batch_first=True, bias=True, num_layers=2)
        
        # Decoder input size is embedding_dim + context_vector size (state_size)
        self.decoder = nn.LSTM(input_size=self.embedding_dim + self.state_size, hidden_size=self.state_size, batch_first=True, bias=True, num_layers=2)

        self.encoder_embedding = nn.Embedding(num_embeddings=src_vectorizer.vocab_size, embedding_dim=self.embedding_dim)
        self.decoder_embedding = nn.Embedding(num_embeddings=trg_vectorizer.vocab_size, embedding_dim=self.embedding_dim)

        # Attention output must be 1 to calculate scores
        self.attention = nn.Linear(2 * self.state_size, 1)
        
        # Added the final fully connected projection layer
        self.fc_out = nn.Linear(self.state_size, trg_vectorizer.vocab_size)

    # Added max_trg_len as an argument so the inference loop knows when to stop
    def forward(self, encoder_input, decoder_input=None, training=False, max_trg_len=50):
        batch_size = encoder_input.size(0)
        
        encoder_embedded = self.encoder_embedding(encoder_input)
        encoder_output, (hidden, cell) = self.encoder_lstm(encoder_embedded)
        prev_decoder_state = (hidden, cell)
        
        # We need the encoder sequence length to properly shape our attention tensors
        seq_len = encoder_output.size(1)

        if training:
            # List to store predictions for the whole sentence
            all_predictions = [] 
            
            for word_idx in range(decoder_input.size(1)):
                decoder_word_input = decoder_input[:, word_idx].unsqueeze(1)
                
                # The Shape Fix. Expand the decoder hidden state from [1, batch, 128] 
                # to [batch, seq_len, 128] so it matches encoder_output for concatenation.
                hidden_expanded = prev_decoder_state[0].transpose(0, 1).repeat(1, seq_len, 1)
                
                attention_input = torch.cat((hidden_expanded, encoder_output), dim=2)
                attention_weights = torch.softmax(self.attention(attention_input), dim=1)
                context_vector = torch.sum(attention_weights * encoder_output, dim=1, keepdim=True)

                decoder_embedded = self.decoder_embedding(decoder_word_input)
                decoder_embedded = torch.cat((context_vector, decoder_embedded), dim=2)
                decoder_output, (hidden, cell) = self.decoder(decoder_embedded, prev_decoder_state)
                prev_decoder_state = (hidden, cell)

                # Pass through final projection layer and save it
                prediction = self.fc_out(decoder_output)
                all_predictions.append(prediction)
                
            # Concatenate list of [batch, 1, vocab_size] into [batch, seq_len, vocab_size]
            return torch.cat(all_predictions, dim=1), prev_decoder_state

        else:
            start_token = self.trg_vectorizer.word_to_index['startseq']
            # Made device dynamic so it works on GPU if your inputs are on GPU
            decoder_word_input = torch.full((batch_size, 1), start_token, dtype=torch.long)
            decoded_words = []

            for word_idx in range(max_trg_len):
                # The Shape Fix for inference
                hidden_expanded = prev_decoder_state[0].transpose(0, 1).repeat(1, seq_len, 1)

                attention_input = torch.cat((hidden_expanded, encoder_output), dim=2)
                attention_weights = torch.softmax(self.attention(attention_input), dim=1)
                context_vector = torch.sum(attention_weights * encoder_output, dim=1, keepdim=True)

                decoder_embedded = self.decoder_embedding(decoder_word_input)
                decoder_embedded = torch.cat((context_vector, decoder_embedded), dim=2)
                decoder_output, (hidden, cell) = self.decoder(decoder_embedded, prev_decoder_state)
                prev_decoder_state = (hidden, cell)

                # Pass through final layer BEFORE doing argmax
                prediction = self.fc_out(decoder_output)
                predicted_word_idx = torch.argmax(prediction[:, -1, :], dim=1).unsqueeze(1)
                
                decoder_word_input = predicted_word_idx
                decoded_words.append(predicted_word_idx)
                
            # Un-indented the return statement so the loop actually finishes!
            # Returns a tensor of shape [batch, max_trg_len] containing predicted token IDs
            return torch.cat(decoded_words, dim=1), prev_decoder_state

In [17]:
model = LSTMModel(src_vectorizer, trg_vectorizer)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [18]:
# Convert TensorFlow tensors -> NumPy arrays -> PyTorch tensors
enc_in_pt = torch.tensor(encoder_input.numpy(), dtype=torch.long)
dec_in_pt = torch.tensor(decoder_input.numpy(), dtype=torch.long)
dec_tar_pt = torch.tensor(decoder_target.numpy(), dtype=torch.long)

# Create the PyTorch Dataset and DataLoader
batch_size = 64
dataset = TensorDataset(enc_in_pt, dec_in_pt, dec_tar_pt)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [19]:
def train_model(model, train_loader, optimizer, criterion, num_epochs=10):
    model.train() # Turns on your self.training flag inside the model!

    for epoch in range(num_epochs):
        epoch_loss = 0
        
        for batch_idx, (src_data, trg_in_data, trg_out_data) in enumerate(train_loader):
            optimizer.zero_grad()
            
            # Forward pass uses 'trg_in_data' (which has startseq)
            output, _ = model(encoder_input=src_data, decoder_input=trg_in_data, training=True)
            
            # Reshape for CrossEntropyLoss
            output_dim = output.shape[-1]
            output = output.reshape(-1, output_dim) 
            
            # Target uses 'trg_out_data' (which has endseq)
            target = trg_out_data.reshape(-1) 
            
            loss = criterion(output, target)
            
            # 5. Backward pass
            loss.backward()
            
            # 6. Gradient Clipping (Highly recommended for LSTMs to prevent exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # 7. Update weights
            optimizer.step()
            
            epoch_loss += loss.item()
            
        # Print average loss for the epoch
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss / len(train_loader):.4f}")

In [20]:
# Keras usually uses 0 for padding index
pad_idx = 0 
if '' in trg_vectorizer.word_to_index:
    pad_idx = trg_vectorizer.word_to_index['']
train_model(model, train_loader, optimizer, nn.CrossEntropyLoss(ignore_index=pad_idx), num_epochs=10)

Epoch [1/10] | Loss: 5.2418
Epoch [2/10] | Loss: 4.0646
Epoch [3/10] | Loss: 3.4972
Epoch [4/10] | Loss: 3.0883
Epoch [5/10] | Loss: 2.7535
Epoch [6/10] | Loss: 2.4706
Epoch [7/10] | Loss: 2.2258
Epoch [8/10] | Loss: 2.0060
Epoch [9/10] | Loss: 1.8136
Epoch [10/10] | Loss: 1.6389


In [26]:
def translate_sentence(model, sentence, src_vectorizer, trg_vectorizer, max_len=50):
    # 1. Put the model in evaluation mode (turns off dropout, batchnorm, etc.)
    model.eval()

    # 2. Preprocess the raw string using your Keras vectorizer
    # We wrap it in a list because the vectorizer expects a batch
    input_vector = src_vectorizer(np.array([sentence]))
    
    # 3. Convert to PyTorch tensor and move to GPU/CPU
    input_tensor = torch.tensor(input_vector.numpy(), dtype=torch.long)

    # 4. Turn off gradient tracking (saves memory and speeds up inference)
    with torch.no_grad():
        # Pass training=False to trigger your custom inference loop!
        output_tokens, _ = model(encoder_input=input_tensor, training=False, max_trg_len=max_len)

    # output_tokens shape is [1, max_len]. We squeeze it to a 1D list of numbers.
    output_tokens = output_tokens.squeeze().cpu().numpy()

    # 5. Convert token IDs back to words
    vocab = trg_vectorizer.get_vocabulary()
    translated_words = []
    
    for token_id in output_tokens:
        word = vocab[token_id]
        
        # Stop translating if the model outputs the end token
        if word == 'endseq':
            break
            
        # Ignore padding or unknown tokens in the final output
        if word not in ['', '<PAD>', '[UNK]', 'startseq']:
            translated_words.append(word)

    # Join the list of words into a single string
    return " ".join(translated_words)

In [27]:
test_sentences = [
    "I love you.",
    "He is a good boy.",
    "Run!",
    "I am very hungry."
]

print("\n--- Translation Test ---")
for eng_text in test_sentences:
    fra_translation = translate_sentence(
        model=model, 
        sentence=eng_text, 
        src_vectorizer=src_vectorizer, 
        trg_vectorizer=trg_vectorizer, 
    )
    print(f"English: {eng_text}")
    print(f"French:  {fra_translation}\n")


--- Translation Test ---
English: I love you.
French:  je vous prie

English: He is a good boy.
French:  c'est un livre

English: Run!
French:  fuyons

English: I am very hungry.
French:  je suis très timide

